In [1]:
import torch
import torch.nn as nn
import h5py
import numpy as np
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from transformers import AutoTokenizer
from torch_geometric.data import Data
from torch_geometric.utils import add_self_loops, dense_to_sparse
import time
import math
import random
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.auto import tqdm
import json
import evaluate as hf_evaluate
import os

# ==================================================================================
# CONFIGURATION
# ==================================================================================

H5_FILE_PATH = "/home/poorna/data/eeg_dataset_1400_multilabel.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
OBJECT_MAPPING_FILE = "/home/poorna/data/object_id_to_name_blip.json"

BATCH_SIZE = 16
EPOCHS = 30

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
PAD_ID = tokenizer.pad_token_id
SOS_ID = tokenizer.cls_token_id
EOS_ID = tokenizer.sep_token_id
TEXT_VOCAB_SIZE = tokenizer.vocab_size

# Labels
NUM_COLORS = 9       
NUM_OBJECTS = 6      
TOTAL_META_DIM = NUM_COLORS + NUM_OBJECTS 

/home/poorna/venvs/torch/lib64/python3.11/site-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.sparse import csr_matrix, issparse


Using device: cuda


In [2]:
# ==================================================================================
# 1. ROBUST GRAPH GENERATION (STANDARD 10-10 SYSTEM)
# ==================================================================================

def get_1010_geometric_graph(k_neighbors=6):
    """
    Creates a K-Nearest Neighbor graph based on the specific 10-10 coordinates provided.
    Converts Polar (Theta, Radius) to Cartesian (X, Y) to compute distances.
    """
    # Format: [Channel Index (0-based), Theta (deg), Radius (0-1)]
    # derived from your provided list
    coords_raw = [
        (0, -18, 0.51111), (1, 0, 0.51111), (2, 18, 0.51111), (3, -23, 0.41111), 
        (4, 23, 0.41111), (5, -54, 0.51111), (6, -49, 0.41667), (7, -39, 0.33333), 
        (8, -22, 0.27778), (9, 0, 0.25556), (10, 22, 0.27778), (11, 39, 0.33333), 
        (12, 49, 0.41667), (13, 54, 0.51111), (14, -72, 0.51111), (15, -69, 0.39444), 
        (16, -62, 0.27778), (17, -45, 0.17778), (18, 0, 0.12778), (19, 45, 0.17778), 
        (20, 62, 0.27778), (21, 69, 0.39444), (22, 72, 0.51111), (23, -90, 0.51111), 
        (24, -90, 0.38333), (25, -90, 0.25556), (26, -90, 0.12778), (27, 90, 0.0), 
        (28, 90, 0.12778), (29, 90, 0.25556), (30, 90, 0.38333), (31, 90, 0.51111), 
        (32, -105, 0.51111), (33, -111, 0.39444), (34, -118, 0.27778), (35, -135, 0.17778), 
        (36, 180, 0.12778), (37, 135, 0.17778), (38, 118, 0.27778), (39, 111, 0.39444), 
        (40, 105, 0.51111), (41, -120, 0.51111), (42, -131, 0.41667), (43, -141, 0.33333), 
        (44, -158, 0.27778), (45, 180, 0.25556), (46, 158, 0.27778), (47, 141, 0.33333), 
        (48, 131, 0.41667), (49, 120, 0.51111), (50, -135, 0.51111), (51, -147, 0.46838), 
        (52, -157, 0.41111), (53, 180, 0.38333), (54, 157, 0.41111), (55, 147, 0.46838), 
        (56, 135, 0.51111), (57, -150, 0.51111), (58, -165, 0.51111), (59, 180, 0.51111), 
        (60, 165, 0.51111), (61, 150, 0.51111)
    ]

    num_nodes = 62
    pos = np.zeros((num_nodes, 2))

    # Convert Polar to Cartesian (X, Y)
    for idx, theta, radius in coords_raw:
        rad = np.deg2rad(theta)
        # Assuming standard EEG projection: 
        # 0 deg is North (Nose), 90 is Right, -90 is Left
        # x = r * sin(theta), y = r * cos(theta) 
        # But based on your T7 (-90) and T8 (90), this is standard unit circle with rotation
        x = radius * np.sin(rad)
        y = radius * np.cos(rad)
        pos[idx] = [x, y]

    # Calculate Euclidean Distance Matrix
    dist_matrix = np.zeros((num_nodes, num_nodes))
    for i in range(num_nodes):
        for j in range(num_nodes):
            dist_matrix[i, j] = np.linalg.norm(pos[i] - pos[j])

    # Build KNN Graph
    edge_list = []
    for i in range(num_nodes):
        # Get indices of k+1 nearest nodes (including self)
        # argsort gives ascending order
        nearest_indices = np.argsort(dist_matrix[i])[:k_neighbors + 1]
        
        for neighbor in nearest_indices:
            if i != neighbor: # Don't add self-loop yet (GCN adds it later or manually)
                edge_list.append([i, neighbor])
                
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    
    # Create uniform edge weights for now
    edge_attr = torch.ones(edge_index.shape[1], dtype=torch.float)
    
    return edge_index.to(device), edge_attr.to(device)

In [3]:
# ==================================================================================
# DATA HANDLING
# ==================================================================================

def create_stratified_split(total_samples, group_size=5):
    num_groups = total_samples // group_size
    train_indices = []
    val_indices = []
    test_indices = []
    
    for group_idx in range(num_groups):
        start_idx = group_idx * group_size
        group_indices = list(range(start_idx, start_idx + group_size))
        train_indices.extend(group_indices[:3])  # 3 Train
        val_indices.append(group_indices[3])     # 1 Val
        test_indices.append(group_indices[4])    # 1 Test
    
    # Handle remainder
    remainder = total_samples % group_size
    if remainder > 0:
        start_idx = num_groups * group_size
        remainder_indices = list(range(start_idx, total_samples))
        train_indices.extend(remainder_indices) # Just put remainder in train to be safe

    print(f"Split: Train {len(train_indices)}, Val {len(val_indices)}, Test {len(test_indices)}")
    return train_indices, val_indices, test_indices

class EEGMetaTextH5Dataset(Dataset):
    def __init__(self, h5_path):
        self.h5_path = h5_path
        self.h5_file = None
        with h5py.File(self.h5_path, 'r') as f:
            self.n_samples = f['eeg'].shape[0]

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        if self.h5_file is None:
            self.h5_file = h5py.File(self.h5_path, 'r')

        eeg = torch.from_numpy(self.h5_file['eeg'][idx].astype(np.float32))
        meta = torch.from_numpy(self.h5_file['metadata'][idx].astype(np.float32))
        text = torch.from_numpy(self.h5_file['input_ids'][idx].astype(np.int64))
        return eeg, meta, text

def collate_multimodal_batch(batch):
    eeg_list, meta_list, text_list = [], [], []
    for eeg, meta, txt in batch:
        eeg_list.append(eeg)
        meta_list.append(meta)
        text_list.append(txt)
    eeg_batch = torch.stack(eeg_list, dim=0)
    meta_batch = torch.stack(meta_list, dim=0)
    text_padded = pad_sequence(text_list, batch_first=True, padding_value=PAD_ID)
    return eeg_batch.float(), meta_batch.float(), text_padded

In [4]:
# ==================================================================================
# CORRECTED MODEL ARCHITECTURE
# ==================================================================================

class SpatioTemporalEEGEncoder(nn.Module):
    def __init__(self, num_channels=62, enc_hidden=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.num_channels = num_channels
        self.gcn1 = GCNConv(num_channels, enc_hidden)
        self.gcn2 = GCNConv(enc_hidden, enc_hidden)
        self.rnn = nn.GRU(enc_hidden, enc_hidden, num_layers,
                          bidirectional=True, dropout=dropout if num_layers > 1 else 0,
                          batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, eeg, edge_index, edge_attr):
        batch_size = eeg.shape[0]
        num_timesteps = eeg.shape[2]

        # Expand graph for batch
        batch_edge_index = edge_index.repeat(1, batch_size)
        batch_edge_attr = edge_attr.repeat(batch_size)
        batch_offset = torch.arange(batch_size, device=eeg.device) * self.num_channels
        batch_edge_index = batch_edge_index + batch_offset.repeat_interleave(edge_index.shape[1])

        eeg_reshaped = eeg.permute(0, 2, 1).reshape(-1, self.num_channels)

        x = F.relu(self.gcn1(eeg_reshaped, batch_edge_index, batch_edge_attr))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, batch_edge_index, batch_edge_attr))

        temporal_features = x.reshape(batch_size, num_timesteps, -1)
        encoder_outputs, encoder_hidden = self.rnn(temporal_features)
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        return encoder_outputs, encoder_hidden

class LuongAttention(nn.Module):
    def __init__(self, enc_dim, dec_dim):
        super().__init__()
        self.attn = nn.Linear(enc_dim, dec_dim)

    def forward(self, decoder_hidden, encoder_outputs):
        # decoder_hidden: [1, batch, dec_dim] (usually top layer)
        scores = torch.bmm(decoder_hidden.permute(1, 0, 2), self.attn(encoder_outputs).permute(1, 2, 0))
        attn_weights = F.softmax(scores, dim=2)
        context = torch.bmm(attn_weights, encoder_outputs.permute(1, 0, 2))
        return context, attn_weights.squeeze(1)

class MetadataEncoder(nn.Module):
    def __init__(self, num_colors, num_objects, color_feature_dim=32, object_feature_dim=32):
        super().__init__()
        self.color_processor = nn.Sequential(
            nn.Linear(num_colors, 64), nn.ReLU(), nn.Linear(64, color_feature_dim)
        )
        self.object_processor = nn.Sequential(
            nn.Linear(num_objects, 64), nn.ReLU(), nn.Linear(64, object_feature_dim)
        )
        self.output_dim = color_feature_dim + object_feature_dim

    def forward(self, metadata):
        c = self.color_processor(metadata[:, :NUM_COLORS].float())
        o = self.object_processor(metadata[:, NUM_COLORS:].float())
        return torch.cat([c, o], dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hidden, dec_hidden, meta_features_dim, num_layers, pad_id, dropout):
        super().__init__()
        self.vocab_size = vocab_size
        self.dec_hidden = dec_hidden
        self.num_layers = num_layers
        enc_dim = enc_hidden * 2

        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.attention = LuongAttention(enc_dim, dec_hidden)

        # FIX: Removed meta_features_dim from rnn_input
        self.rnn_input_dim = emb_dim + enc_dim + enc_dim 
        self.rnn = nn.GRU(self.rnn_input_dim, dec_hidden, num_layers, dropout=dropout if num_layers > 1 else 0)

        self.fc_out = nn.Linear(dec_hidden, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.bridge = nn.Linear(enc_dim, dec_hidden)
        
        # FIX: Projector to fuse metadata into hidden state initialization
        self.init_projector = nn.Linear(dec_hidden + meta_features_dim, dec_hidden)

    def init_hidden(self, encoder_hidden, meta_features):
        # 1. Process Encoder Hidden
        hidden = encoder_hidden.view(self.num_layers, 2, encoder_hidden.size(1), -1)
        last_layer_hidden = hidden[-1]
        encoder_hidden_cat = torch.cat((last_layer_hidden[0], last_layer_hidden[1]), dim=1)
        bridged_hidden = torch.tanh(self.bridge(encoder_hidden_cat))
        
        # 2. Fuse with Metadata (Crucial Fix)
        combined = torch.cat([bridged_hidden, meta_features], dim=1)
        init_hidden_state = torch.tanh(self.init_projector(combined))
        
        # Repeat for all layers
        return init_hidden_state.unsqueeze(0).repeat(self.num_layers, 1, 1)

    def forward(self, token, decoder_hidden, encoder_outputs, global_eeg_context):
        token = token.unsqueeze(0)
        embedded = self.dropout(self.embedding(token))
        context, attn_weights = self.attention(decoder_hidden[-1].unsqueeze(0), encoder_outputs)
        
        context_permuted = context.permute(1, 0, 2)
        global_eeg_context_unsqueezed = global_eeg_context.unsqueeze(0)

        # FIX: Metadata removed from here to prevent leakage/cheating
        rnn_input = torch.cat((
            embedded,
            context_permuted,
            global_eeg_context_unsqueezed
        ), dim=2)

        output, hidden = self.rnn(rnn_input, decoder_hidden)
        prediction = self.fc_out(output.squeeze(0))

        return prediction, hidden, context.squeeze(1)

class Seq2Seq(nn.Module):
    def __init__(self, text_vocab_size, num_colors, num_objects, enc_hidden=256, dec_hidden=256,
                 pad_id=0, dropout=0.2, emb_dim=256, dec_layers=2):
        super().__init__()
        self.encoder = SpatioTemporalEEGEncoder(enc_hidden=enc_hidden, dropout=dropout, num_layers=dec_layers)
        self.meta_encoder = MetadataEncoder(num_colors, num_objects)
        
        meta_dim = self.meta_encoder.output_dim
        enc_dim = enc_hidden * 2

        self.decoder = Decoder(text_vocab_size, emb_dim, enc_hidden, dec_hidden,
                               meta_dim, dec_layers, pad_id, dropout)

        self.meta_head = nn.Sequential(
            nn.Linear(enc_dim, 256), nn.ReLU(), nn.LayerNorm(256), nn.Dropout(0.3),
            nn.Linear(256, num_colors + num_objects)
        )
        self.num_colors = num_colors
        self.num_objects = num_objects

    def forward(self, eeg, metadata, target_text, edge_index, edge_attr, teacher_forcing_ratio=0.5):
        batch_size = eeg.shape[0]
        target_len = target_text.shape[1]
        vocab_size = self.decoder.vocab_size

        encoder_outputs, encoder_hidden = self.encoder(eeg, edge_index, edge_attr)
        meta_features = self.meta_encoder(metadata)

        # Initialize decoder with fused EEG+Metadata
        decoder_hidden = self.decoder.init_hidden(encoder_hidden, meta_features)

        # Global context for RNN input
        hidden_reshaped = encoder_hidden.view(self.encoder.rnn.num_layers, 2, batch_size, -1)
        last_layer = hidden_reshaped[-1]
        global_eeg_context = torch.cat((last_layer[0], last_layer[1]), dim=1)

        # Auxiliary Tasks
        meta_preds = self.meta_head(global_eeg_context)
        pred_color = meta_preds[:, :self.num_colors]
        pred_object = meta_preds[:, self.num_colors:]

        outputs = torch.zeros(target_len, batch_size, vocab_size).to(eeg.device)
        decoder_input = target_text[:, 0]

        for t in range(1, target_len):
            output, decoder_hidden, _ = self.decoder(
                decoder_input, decoder_hidden, encoder_outputs, global_eeg_context
            )
            outputs[t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            decoder_input = target_text[:, t] if teacher_force else top1

        return outputs[1:].permute(1, 0, 2), pred_color, pred_object

In [5]:
# ==================================================================================
# LOSS AND UTILS
# ==================================================================================

class CurriculumMultiTaskLoss(nn.Module):
    def __init__(self, initial_aux_weight=2.0, min_aux_weight=0.1, decay_epochs=20):
        super().__init__()
        self.initial_aux_weight = initial_aux_weight
        self.min_aux_weight = min_aux_weight
        self.decay_epochs = decay_epochs
        self.current_epoch = 0
    
    def get_aux_weight(self):
        if self.current_epoch >= self.decay_epochs: return self.min_aux_weight
        progress = self.current_epoch / self.decay_epochs
        return self.initial_aux_weight * (self.min_aux_weight / self.initial_aux_weight) ** progress
    
    def forward(self, loss_t, loss_c, loss_o):
        w = self.get_aux_weight()
        return loss_t + w * (loss_c + loss_o), w
    
    def step_epoch(self):
        self.current_epoch += 1

class DiversityLoss(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size
    
    def forward(self, logits):
        probs = F.softmax(logits, dim=-1)
        avg_probs = probs.mean(dim=(0, 1))
        uniform = torch.ones_like(avg_probs) / self.vocab_size
        return F.kl_div(avg_probs.log(), uniform, reduction='batchmean')

def calibrated_multilabel_predict(logits, threshold=0.3, max_labels=3): # Thresh lowered to 0.3
    probs = torch.sigmoid(logits)
    predictions = (probs > threshold).float()
    num_predicted = predictions.sum(dim=1)
    for i in range(predictions.size(0)):
        if num_predicted[i] > max_labels:
            top_k_values, top_k_indices = torch.topk(probs[i], max_labels)
            predictions[i] = 0.0
            predictions[i][top_k_indices] = 1.0
    return predictions

In [6]:
# ==================================================================================
# TRAINING LOOP
# ==================================================================================

def train_one_epoch(model, loader, optimizer, criterion_t, criterion_c, criterion_o, 
                    criterion_d, curriculum, edge_index, edge_attr, div_w=0.01, tf_ratio=0.5):
    model.train()
    total_loss = 0.0
    losses = {'text':0.0, 'color':0.0, 'object':0.0}
    
    pbar = tqdm(loader, desc="Training", leave=False)
    for eeg, meta, txt in pbar:
        eeg, txt, meta = eeg.to(device), txt.to(device), meta.to(device)
        optimizer.zero_grad()

        text_logits, pred_c, pred_o = model(eeg, meta, txt, edge_index, edge_attr, tf_ratio)

        loss_t = criterion_t(text_logits.reshape(-1, text_logits.shape[-1]), txt[:, 1:].reshape(-1))
        loss_c = criterion_c(pred_c, meta[:, :NUM_COLORS].float())
        loss_o = criterion_o(pred_o, meta[:, NUM_COLORS:].float())
        loss_div = criterion_d(text_logits)

        loss, w = curriculum(loss_t, loss_c, loss_o)
        loss += div_w * loss_div

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        losses['text'] += loss_t.item()
        losses['color'] += loss_c.item()
        losses['object'] += loss_o.item()
        
        pbar.set_postfix(loss=loss.item(), txt=loss_t.item(), aux_w=w)
        
    return {k:v/len(loader) for k,v in losses.items()}, total_loss/len(loader), w

@torch.no_grad()
def evaluate(model, loader, criterion_t, criterion_c, criterion_o, criterion_d, 
             curriculum, edge_index, edge_attr, div_w=0.01):
    model.eval()
    total_loss = 0.0
    losses = {'text':0.0, 'color':0.0, 'object':0.0}
    
    for eeg, meta, txt in tqdm(loader, desc="Eval", leave=False):
        eeg, txt, meta = eeg.to(device), txt.to(device), meta.to(device)
        
        text_logits, pred_c, pred_o = model(eeg, meta, txt, edge_index, edge_attr, 0.0)

        loss_t = criterion_t(text_logits.reshape(-1, text_logits.shape[-1]), txt[:, 1:].reshape(-1))
        loss_c = criterion_c(pred_c, meta[:, :NUM_COLORS].float())
        loss_o = criterion_o(pred_o, meta[:, NUM_COLORS:].float())
        
        loss, _ = curriculum(loss_t, loss_c, loss_o)
        loss += div_w * criterion_d(text_logits)

        total_loss += loss.item()
        losses['text'] += loss_t.item()
        losses['color'] += loss_c.item()
        losses['object'] += loss_o.item()

    return {k:v/len(loader) for k,v in losses.items()}, total_loss/len(loader)

@torch.no_grad()
def generate(model, eeg, meta, edge_index, edge_attr, max_len=30):
    model.eval()
    eeg = eeg.unsqueeze(0).to(device)
    meta = meta.unsqueeze(0).to(device)
    
    enc_out, enc_hid = model.encoder(eeg, edge_index, edge_attr)
    meta_feat = model.meta_encoder(meta)
    dec_hid = model.decoder.init_hidden(enc_hid, meta_feat)
    
    # Global context
    hid_reshaped = enc_hid.view(model.encoder.rnn.num_layers, 2, 1, -1)
    last_layer = hid_reshaped[-1]
    global_ctx = torch.cat((last_layer[0], last_layer[1]), dim=1)

    # Aux predictions
    aux = model.meta_head(global_ctx)
    pred_c_ids = calibrated_multilabel_predict(aux[0, :NUM_COLORS].unsqueeze(0), 0.3).nonzero(as_tuple=True)[1].tolist()
    pred_o_ids = calibrated_multilabel_predict(aux[0, NUM_COLORS:].unsqueeze(0), 0.3).nonzero(as_tuple=True)[1].tolist()

    # Decode Text
    curr_token = torch.tensor([SOS_ID], device=device)
    generated = [SOS_ID]
    
    for _ in range(max_len):
        pred, dec_hid, _ = model.decoder(curr_token, dec_hid, enc_out, global_ctx)
        next_token = pred.argmax().item()
        generated.append(next_token)
        if next_token == EOS_ID: break
        curr_token = torch.tensor([next_token], device=device)
        
    return tokenizer.decode(generated, skip_special_tokens=True), pred_c_ids, pred_o_ids

In [7]:
# ==================================================================================
# MAIN
# ==================================================================================

if __name__ == "__main__":
    # 1. Setup Data
    dataset = EEGMetaTextH5Dataset(H5_FILE_PATH)
    train_idx, val_idx, test_idx = create_stratified_split(len(dataset))
    
    train_loader = DataLoader(Subset(dataset, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_multimodal_batch)
    val_loader = DataLoader(Subset(dataset, val_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)
    test_loader = DataLoader(Subset(dataset, test_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_multimodal_batch)

    # 2. Build Graph
    print("\n=== Building 10-10 KNN Graph ===")
    edge_index, edge_attr = get_1010_geometric_graph(k_neighbors=6)
    edge_index, edge_attr = add_self_loops(edge_index, edge_attr=edge_attr, fill_value=1.0)
    print(f"Graph Edges: {edge_index.shape[1]}")

    # 3. Model & Training
    model = Seq2Seq(TEXT_VOCAB_SIZE, NUM_COLORS, NUM_OBJECTS).to(device)
    
    # Pos weights from your previous code
    w_obj = torch.tensor([3.1176, 5.6667, 6.1066, 1.1021, 2.0905, 7.5366]).to(device)
    w_col = torch.tensor([3.1543, 1.2764, 3.9645, 1.7888, 0.8301, 11.2807, 5.2780, 1.0408, 4.4054]).to(device)
    
    crit_t = nn.CrossEntropyLoss(ignore_index=PAD_ID)
    crit_c = nn.BCEWithLogitsLoss(pos_weight=w_col)
    crit_o = nn.BCEWithLogitsLoss(pos_weight=w_obj)
    crit_d = DiversityLoss(TEXT_VOCAB_SIZE).to(device)
    
    curriculum = CurriculumMultiTaskLoss().to(device)
    opt = AdamW(model.parameters(), lr=3e-5, weight_decay=1e-2)
    sched = ReduceLROnPlateau(opt, 'min', patience=3, factor=0.5)

    print("\n=== Starting Training ===")
    best_loss = float('inf')
    
    for epoch in range(1, EPOCHS+1):
        start = time.time()
        
        # FIX: Linear Decay for Teacher Forcing (1.0 -> 0.0)
        tf_ratio = max(0.0, 1.0 - (epoch / 15)) 
        
        train_res, train_loss, aux_w = train_one_epoch(
            model, train_loader, opt, crit_t, crit_c, crit_o, crit_d, curriculum, edge_index, edge_attr, tf_ratio=tf_ratio
        )
        val_res, val_loss = evaluate(
            model, val_loader, crit_t, crit_c, crit_o, crit_d, curriculum, edge_index, edge_attr
        )
        
        curriculum.step_epoch()
        sched.step(val_loss)
        
        print(f"Ep {epoch:02} ({int(time.time()-start)}s) | TF: {tf_ratio:.2f} | AuxW: {aux_w:.2f}")
        print(f"  Train: {train_loss:.4f} (Txt: {train_res['text']:.3f})")
        print(f"  Val:   {val_loss:.4f} (Txt: {val_res['text']:.3f})")
        
        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), "best_eeg_model.pt")
            print("  [*] Model Saved")

    # 4. Inference
    print("\n=== Final Inference ===")
    model.load_state_dict(torch.load("best_eeg_model.pt", map_location=device))
    
    preds, refs = [], []
    for i in range(min(20, len(test_loader.dataset))):
        eeg, meta, txt = test_loader.dataset[i]
        p_txt, p_c, p_o = generate(model, eeg, meta, edge_index, edge_attr)
        ref_txt = tokenizer.decode(txt.tolist(), skip_special_tokens=True)
        
        preds.append(p_txt)
        refs.append(ref_txt)
        
        print(f"\nSample {i}:")
        print(f"  GT:   {ref_txt}")
        print(f"  Pred: {p_txt}")
        print(f"  Cols: {p_c} | Objs: {p_o}")
        
    # Compute Metrics
    bleu = hf_evaluate.load('bleu')
    rouge = hf_evaluate.load('rouge')
    
    print("\nScores:")
    print(bleu.compute(predictions=preds, references=[[r] for r in refs]))
    print(rouge.compute(predictions=preds, references=refs))

Split: Train 16800, Val 5600, Test 5600

=== Building 10-10 KNN Graph ===
Graph Edges: 434

=== Starting Training ===


Training:   0%|          | 0/1050 [00:00<?, ?it/s]

Eval:   0%|          | 0/350 [00:00<?, ?it/s]

Ep 01 (365s) | TF: 0.93 | AuxW: 2.00
  Train: 9.4714 (Txt: 5.329)
  Val:   8.5162 (Txt: 4.459)
  [*] Model Saved


Training:   0%|          | 0/1050 [00:00<?, ?it/s]

...

Training:   0%|          | 0/1050 [00:00<?, ?it/s]

Eval:   0%|          | 0/350 [00:00<?, ?it/s]

Ep 09 (416s) | TF: 0.40 | AuxW: 0.60
  Train: 4.1227 (Txt: 2.905)
  Val:   5.6054 (Txt: 4.382)
  [*] Model Saved


Training:   0%|          | 0/1050 [00:00<?, ?it/s]

KeyboardInterrupt: 